# 05 · 示范不是标签表：行为克隆怎样进入驾驶闭环？

你已经训练过监督学习模型。本课把一个熟悉的训练循环接到真实的 MetaDrive 轨迹上：几何控制器访问状态并发出动作，车辆执行动作后产生下一状态；模型只学习当前状态到专家动作的映射。

**前置课：Unit 02 状态估计。** 本课暂时使用模拟器 privileged truth 来隔离“监督训练和闭环”这个问题；它不是已经接入传感器估计的自动驾驶系统。上一单元的真值状态、参考车道和 `run_episode` 轨迹契约仍是本课的输入边界。

目标是能解释三件事：输入从哪里来、episode 为什么先切分再展平、离线 MSE 为什么不能代替闭环评测。所有运行结果保存在 `artifacts/imitation/`，先收集数据，再训练，再重载 checkpoint。

## 1. 先写数据契约

每个样本的特征是 `(e_y, heading_error, speed, reference_bearing)`：横向误差、相对车道朝向、速度、车辆到前视参考点的方位角。动作是 `(steering, throttle)`，范围为 `[-1,1]`。

这些字段描述控制时刻已经可见的状态和参考。动作、reward、terminal flag、执行后状态以及未来状态都不进入输入；它们可以用于评测，但不能成为输入泄漏。数据还保留 episode id、seed、初始偏移和完整配置。

In [ ]:
from pathlib import Path
import sys
from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "ad_tutorial").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from ad_tutorial.imitation import *

In [ ]:
configs = default_split_configs(horizon=40)
dataset = collect_demonstrations(configs)
print(dataset.manifest())
print("hash:", dataset_hash(dataset))
assert set(FEATURE_NAMES) == {"e_y_m", "heading_error_rad", "speed_mps", "reference_bearing_rad"}
assert not any("action" in name or "reward" in name or "next" in name for name in FEATURE_NAMES)

## 2. 为什么不能按 timestep 随机切分？

同一条轨迹相邻时刻的状态高度相关。若把一条 episode 的前半放训练、后半放测试，MSE 可能很漂亮，但测试状态仍是训练车辆刚刚访问过的局部邻域。这里先按完整 episode 分成 train、validation、test，再把各组展平。

默认 train 偏移为 `±0.1、±0.2m`，validation 是独立 episode 的 `±0.25m`，test 是 `±0.4m`。seed 也分开；地图仍是固定的 `block_sequence/S`，所以 seed 变化用于复现实验条件和初始随机性，不能称为道路泛化。

In [ ]:
train_ids = set(dataset.select("train").unique_episode_ids)
val_ids = set(dataset.select("validation").unique_episode_ids)
test_ids = set(dataset.select("test").unique_episode_ids)
assert not train_ids & val_ids and not train_ids & test_ids and not val_ids & test_ids
print({split: dataset.select(split).unique_episode_ids for split in ("train", "validation", "test")})

## 3. 归一化和最小监督梯度

均值和标准差只能从 train 计算。validation/test 可以使用这些统计量，但不能影响它们。下面的函数仍然是普通的 MSE + Adam；没有 reward，也没有策略梯度。

In [ ]:
train, val, test = (dataset.select(name) for name in ("train", "validation", "test"))
fit = train_behavioral_cloning(train, val, BCTrainingConfig(epochs=60, patience=12))
print(fit.metrics)
curve = pd.DataFrame(fit.history)
display(curve.tail())
curve.plot(x="epoch", y=["train_mse", "val_mse"], grid=True)
plt.show()
assert np.isfinite(curve[["train_mse", "val_mse"]].to_numpy()).all()

## 4. 一条样本的可审计链

下面只展开一条真实专家 episode：原始 `trace` → 四维 feature → train-only normalization → MLP prediction，并和该时刻的 expert command 并排显示。这样可以检查字段来源和数量级，而不是把整个过程藏在一次黑盒调用里。

In [ ]:
episode = dataset.episodes[0]
raw_trace = episode["trace"]
one_features = features_from_trace(raw_trace)
one_actions = actions_from_trace(raw_trace)
one_normalized = fit.normalizer.transform(one_features)
with torch.inference_mode():
    one_predictions = fit.model(torch.from_numpy(one_normalized)).numpy()
audit = pd.DataFrame({
    "e_y_m": one_features[:, 0],
    "heading_error_rad": one_features[:, 1],
    "speed_mps": one_features[:, 2],
    "reference_bearing_rad": one_features[:, 3],
    "expert_steering": one_actions[:, 0],
    "model_steering": one_predictions[:, 0],
    "expert_throttle": one_actions[:, 1],
    "model_throttle": one_predictions[:, 1],
    "steering_error": one_predictions[:, 0] - one_actions[:, 0],
    "throttle_error": one_predictions[:, 1] - one_actions[:, 1],
})
for index, feature_name in enumerate(FEATURE_NAMES):
    audit[f"normalized_{feature_name}"] = one_normalized[:, index]
print("episode:", episode["episode_id"], "config seed/offset:",
      episode["config"]["seed"], episode["config"]["initial_lateral_offset_m"])
display(pd.DataFrame(raw_trace)[["before_x_m", "before_y_m", "before_heading_rad",
                                "before_lateral_error_m", "before_speed_mps",
                                "reference_x_m", "reference_y_m", "reference_heading_rad"]].head(5))
display(audit.head(5))
assert one_features.shape[1] == 4 and one_actions.shape[1] == 2

## 5. 一个可运行的线性基线

用 `e_y` 加常数列对两个 expert action 分量分别做闭式最小二乘。它只作为 validation 对照，不参与 MLP 的 checkpoint 选择；如果四特征 MLP 没有超过它，保留这个结果并先查数据和实验条件。

In [ ]:
design_train = np.c_[np.ones(len(train.features)), train.features[:, 0]]
design_val = np.c_[np.ones(len(val.features)), val.features[:, 0]]
linear_coef = np.linalg.lstsq(design_train, train.actions, rcond=None)[0]
linear_val_prediction = np.clip(design_val @ linear_coef, -1, 1)
linear_val_mse = float(np.mean((linear_val_prediction - val.actions) ** 2))
print({"e_y_only_linear_validation_mse": linear_val_mse,
       "four_feature_mlp_validation_mse": fit.metrics["validation_mse"]})

## 6. checkpoint 是接口边界

保存模型时也保存 train-only normalization、特征顺序和动作顺序。重新加载后的 policy 实现 `control(observation, reference)`；`driving.run_episode` 会把它的动作裁剪后传给 `env.step`，所以这一步才是从离线数组回到动力学的连接。

In [ ]:
checkpoint = ROOT / "artifacts" / "imitation" / "lesson05_checkpoint.pt"
save_checkpoint(checkpoint, fit)
reloaded = load_checkpoint(checkpoint)
a = fit.model(torch.from_numpy(fit.normalizer.transform(train.features))).detach().numpy()
b = reloaded.model(torch.from_numpy(reloaded.normalizer.transform(train.features))).detach().numpy()
assert np.allclose(a, b)
print("checkpoint reload action max diff:", np.max(np.abs(a - b)))

## 7. 练习与推理答案

练习：把输入拆成只有 `e_y` 的线性模型；再把它和四特征 MLP 的 validation MSE 对照。预测哪个会更好，并说明理由。然后画出 test 的动作误差直方图。

答案：只有横向误差无法区分“车身朝向已经偏了”和“车辆正在回正”的状态，也无法表达前视点的当前方位，因此通常会损失信息。四特征模型若 MSE 较小，只能说明在离线 test 状态上动作接近专家；它还没有证明自己在 rollout 中访问到的新状态上能恢复。若你的结果相反，保留结果并检查 seed、数据切分和字段来源。

选读：阅读 [DAgger](https://arxiv.org/abs/1011.0686) Introduction 中关于 learner-induced distribution shift 的段落。用本课的偏移量写出“专家访问的状态”和“模型自己访问的状态”各一个例子。这里不实现 DAgger；它是下一阶段的研究阅读方向。